# Lab 2: Reading substance-use measures

### Learning objectives

By the end of Lab 2, you should be able to:

1. Explain why a variable should not be interpreted from its name alone.
2. Use official ABCD documentation as part of responsible measure interpretation.
3. Use documentation to distinguish a variable's scientific role, item or derived status, ABCD storage data type, level of measurement, analytic treatment, and pandas dtype, and use that information to select an appropriate initial summary or visualization.
4. Use frequency tables and bar charts to inspect categorical variables.
5. Use descriptive statistics, a histogram, and a boxplot to inspect a documented quantitative toxicology variable.
6. Identify potential outliers as values for review, not automatic removal.
7. Use variable classifications to select an appropriate initial visualization for one or two variables and explain what the resulting display can and cannot establish.

### Course note: Python in DSARM

DSARM uses Python as a tool for biomedical data literacy. Our main goal is not Python syntax practice for its own sake. Instead, we use simple code to inspect how biomedical measures are stored, summarized, and interpreted.

### A note about using Copilot in this lab

Copilot can help you explain concepts, compare measure types, and restate what code is doing in plain language. Use it to support your learning, not to skip the reasoning steps in the lab. The Copilot prompts in this notebook are conceptual on purpose.


### Dataset provenance

The lab files are synthetic instructional data modeled on real ABCD table and variable structures. Participants are approximately 18 years old at session 07, 19 years old at session 08, and 21 years old at session 10. These session labels establish the simulated longitudinal sequence used throughout the labs.

These are not official ABCD observations, official ABCD waves, or national prevalence estimates.

## 1. Welcome, lab goals, and measure-reading mindset

Lab 1 focused on basic Python objects, CSV files, DataFrames, row units, identifiers, and simple counts. Lab 2 builds on that foundation. We now start reading substance-use measures with simple exploratory data analysis, or EDA.

In this course, reading a measure is not the same thing as naming a variable. A variable name can suggest a topic, but careful interpretation still depends on documentation, table context, response options, and missingness patterns.

As you work through this notebook, keep one question in mind: what does this measure help us see, and what does it still leave uncertain?


## 2. Setup and documentation orientation

We will load four Lab 2 data tables as DataFrames. DEAP is the authoritative source for official ABCD variable wording and metadata classifications; the synthetic CSV files provide the instructional values used for coding practice.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
su_y_sui = None
su_y_mjprob = None
su_y_hairtox = None
su_y_dyn = None

sui_path = Path("../data/labs_01_03/su_y_sui.csv")
mjprob_path = Path("../data/labs_01_03/su_y_mjprob.csv")
hairtox_path = Path("../data/labs_01_03/su_y_hairtox.csv")
dyn_path = Path("../data/labs_01_03/su_y_dyn.csv")

if sui_path.exists():
    su_y_sui = pd.read_csv(sui_path)
if mjprob_path.exists():
    su_y_mjprob = pd.read_csv(mjprob_path)
if hairtox_path.exists():
    su_y_hairtox = pd.read_csv(hairtox_path)
if dyn_path.exists():
    su_y_dyn = pd.read_csv(dyn_path)

lab2_tables = {
    "su_y_sui": su_y_sui,
    "su_y_mjprob": su_y_mjprob,
    "su_y_hairtox": su_y_hairtox,
    "su_y_dyn": su_y_dyn,
}

summary_rows = []
for table_name, dataframe in lab2_tables.items():
    if dataframe is not None:
        summary_rows.append(
            {
                "table_name": table_name,
                "rows": dataframe.shape[0],
                "columns": dataframe.shape[1],
            }
        )

table_summary = pd.DataFrame(summary_rows)
display(table_summary)

### What to notice

The summary table tells us which tables were loaded and how large they are. This is a quick orientation step before we inspect variables inside each table.

Official ABCD documentation is where researchers confirm variable meaning, response options, branching logic, missingness, table structure, and version information. The notebook can show what values appear in the synthetic dataset, but documentation is needed before interpreting what those values mean.

Use DEAP to verify official ABCD variable wording, response coding, `type_var`, `type_data`, and `type_level`. A CSV or pandas dtype shows how values appear in this instructional file; it does not override the official DEAP metadata. If the representations differ, record the difference and investigate it rather than silently changing either source.

### Evidence labels used in this lab

| Label | What it establishes |
|---|---|
| **DEAP** | Variable wording, response coding, respondent, storage data type, variable type, and level of measurement |
| **DOC** | Instrument purpose, administration, reference period, and procedures |
| **SCORE** | Inputs and rules for a documented score |
| **LITERATURE** | External scientific context |
| **RESEARCHER DECISION** | An analytic choice not established by official documentation |



## 3. From DEAP metadata to an initial EDA choice

Lab 1 introduced variable names as structural clues and pandas dtypes as software representations. Exercise 2 introduced the documentation needed to interpret variables. In this section, **DEAP is the source of truth for official ABCD metadata**.

| Description | Question it answers |
|---|---|
| Scientific role | What kind of evidence does the variable provide? |
| DEAP variable type (`type_var`) | Is it documented as an item, summary score, or another variable role? |
| DEAP data type (`type_data`) | How does DEAP document the stored value? |
| DEAP level (`type_level`) | Is the documented level nominal, ordinal, interval, or ratio? |
| Pandas dtype | How did pandas represent this column after import? |
| Analytic treatment | How should the values be handled for this question, based on documentation and coding? |

A variable name, observed value, or pandas dtype may provide a clue, but none replaces DEAP. Analytic treatment and EDA choice require both the documented metadata and the meaning of the recorded values.


### Demo: one DEAP record, several distinct descriptions

Exercise 2 examined `su_y_drgprob_008`, a youth DAPI item about needing more drugs to obtain the same effect during the past six months. DEAP documents it as `type_var: item`, `type_data: character`, and `type_level: nominal`. Its stored codes represent response categories; for example, code `4` represents `10+`, not exactly four events.

The table keeps the official metadata separate from the scientific role and the defensible initial EDA.


In [ ]:
# Run this completed example. No code entry is required.
exercise2_deap_example = pd.DataFrame(
    {
        "description": [
            "Scientific role",
            "DEAP type_var",
            "DEAP type_data",
            "DEAP type_level",
            "What stored code 4 represents",
            "Defensible analytic treatment",
            "Appropriate initial EDA",
        ],
        "su_y_drgprob_008": [
            "youth DAPI problem/consequence item",
            "item",
            "character",
            "nominal",
            "the response category 10+, not exactly four events",
            "categorical, with categories displayed in frequency order",
            "frequency table and ordered bar chart",
        ],
    }
)

display(exercise2_deap_example)

### What to notice

- `type_var`, `type_data`, and `type_level` describe different features of the same variable.
- Numeric-looking codes may still be documented as character categories.
- Analytic treatment can require attention to the displayed response order even when DEAP lists the official level as nominal.
- A pandas dtype describes software representation; it does not determine scientific meaning or measurement level.
- Exercise 2 used DAPI variables from `su_y_drgprob`. This lab uses MAPI variables from `su_y_mjprob`; do not transfer the DAPI scoring rule to MAPI.

> **Further learning with Copilot:** Ask Copilot: Explain why DEAP data type, DEAP variable type, DEAP level of measurement, analytic treatment, and pandas dtype answer different questions.


### Your Turn: use DEAP, then compare with pandas

Use [DEAP](https://abcd.deapscience.com/#/my-datasets/create-dataset) to investigate:

- `su_y_mjprob_004`
- `su_y_dyn__mj_lt_indicator`

Complete the Markdown record below. Then run the completed dtype-check cell that follows; **you do not need to enter or change any code in that cell**.

| Field | `su_y_mjprob_004` | `su_y_dyn__mj_lt_indicator` |
|---|---|---|
| DEAP label or wording | Write here | Write here |
| Respondent and reference period | Write here | Write here |
| DEAP `type_var` | Write here | Write here |
| DEAP `type_data` | Write here | Write here |
| DEAP `type_level` | Write here | Write here |
| What the recorded values represent | Write here | Write here |
| Appropriate initial EDA | Write here | Write here |
| One important limitation | Write here | Write here |

### Prepare for the Canvas reflection: metadata to EDA

Use your completed DEAP record and the pandas dtype output to prepare for these questions:

1. For each variable, what are its documented variable role, data type, and level of measurement, and which initial EDA would you select?
2. Why can a frequency table and bar chart be appropriate for both variables even though one is an item and the other is a summary score?
3. `su_y_mjprob_004` is imported by pandas as `int64`, while DEAP documents character, nominal categories. Why does the pandas dtype not override DEAP?
4. How could treating the codes for `su_y_mjprob_004` as exact event counts produce a misleading statistic or visualization?
5. Name one important interpretive limitation of each variable.

These questions are assessed in Canvas; no additional notebook response is required.

In [ ]:
# Run this completed cell. No code entry is required.
# These pandas dtypes show software representation; they do not supply the DEAP answers.

classification_dtype_check = pd.DataFrame(
    {
        "variable": [
            "su_y_mjprob_004",
            "su_y_dyn__mj_lt_indicator",
        ],
        "pandas_dtype": [
            str(su_y_mjprob["su_y_mjprob_004"].dtype),
            str(su_y_dyn["su_y_dyn__mj_lt_indicator"].dtype),
        ],
    }
)

display(classification_dtype_check)

## 4. Frequency tables and bar charts for categorical variables

When documentation identifies values as categories, frequency tables and bar charts provide a useful initial view of which categories appear and how many records fall in each one. In pandas, `value_counts()` performs this orientation step. A numeric-looking code is not evidence that arithmetic summaries are meaningful.


### Demo

We will inspect `su_y_sui__use__mj__puff_001__l`, a youth self-report cannabis-use item from `su_y_sui`. Documentation identifies it as a categorical report of whether any THC-containing cannabis product was used since the previous visit. That documented meaning—not its pandas dtype—supports an initial frequency table and bar chart.


In [ ]:
sui_use_var = "su_y_sui__use__mj__puff_001__l"

sui_use_counts = su_y_sui[sui_use_var].value_counts(dropna=False)
sui_use_proportions = su_y_sui[sui_use_var].value_counts(normalize=True, dropna=False)

sui_use_freq_table = pd.DataFrame(
    {
        "count": sui_use_counts,
        "proportion": sui_use_proportions,
    }
)

print("Counts:")
print(sui_use_counts)
print()
print("Proportions:")
print(sui_use_proportions)
print()
display(sui_use_freq_table)

sui_use_plot_labels = ["Missing" if pd.isna(value) else str(value) for value in sui_use_counts.index]

plt.figure(figsize=(6, 4))
plt.bar(sui_use_plot_labels, sui_use_counts.values)
plt.title("su_y_sui__use__mj__puff_001__l")
plt.xlabel("Observed value")
plt.ylabel("Count")
plt.show()


### What to notice

`value_counts()` counts the values that appear in a column.

`dropna=False` keeps missing values visible in the output. Missing values are part of the data structure and should not disappear automatically.

`normalize=True` gives proportions instead of raw counts.

The bar chart helps us see the size of each observed category quickly. Documentation is still needed to interpret the categories, reference period, branching, and unavailable values. Counts in this synthetic extract do not automatically estimate prevalence in the ABCD cohort.


### Your Turn

Use the Demo above as your pattern. Your code should follow the same logic and structure, but apply it to the new example below.

Use `su_y_hairtox__rslt__mj__thccooh_cnf` from `su_y_hairtox`.

Create:

- `tox_cnf_counts`
- `tox_cnf_proportions`
- `tox_cnf_freq_table`
- a simple bar chart

Keep all observed values visible. Use DEAP and the hair-toxicology documentation to determine what values such as `Positive`, `Negative`, `666`, or missing represent. Do not recode special values.

After creating the table and chart, be prepared to explain why a categorical summary fits the documented role of a confirmation result and why this result is not equivalent to a quantitative toxicology measurement or a self-report.

> **Further learning with Copilot:** Ask Copilot: Why are counts and bar charts useful for a documented categorical variable, and why do numeric-looking category codes not automatically support means or histograms?


In [ ]:
# Your Turn: inspect a categorical toxicology confirmation variable.

tox_cnf_var = "su_y_hairtox__rslt__mj__thccooh_cnf"

# TODO: Count all observed values, including missing values.
tox_cnf_counts = ______

# TODO: Calculate proportions for all observed values, including missing values.
tox_cnf_proportions = ______

# TODO: Combine the counts and proportions into one DataFrame.
tox_cnf_freq_table = ______

display(tox_cnf_freq_table)

# TODO: Create plot labels that keep missing visible.
tox_cnf_plot_labels = ______

plt.figure(figsize=(6, 4))
# TODO: Create a simple bar chart of the counts.
plt.bar(______, ______)
plt.title("su_y_hairtox__rslt__mj__thccooh_cnf")
plt.xlabel("Observed value")
plt.ylabel("Count")
plt.show()


### Prepare for the Canvas reflection: categorical univariate EDA

Use your frequency table, bar chart, DEAP record, and hair-toxicology documentation to prepare for these questions:

1. What is the documented scientific role and level of measurement of `su_y_hairtox__rslt__mj__thccooh_cnf`?
2. Why are a frequency table and bar chart appropriate initial EDA for this variable?
3. How do `Positive`, `Negative`, code `666`, and a missing value differ? What information would be lost if `666` or missing values were combined with `Negative`?
4. Name one conclusion that confirmation status cannot establish on its own.

Label documentation-based claims as **DEAP** or **DOC**.

These questions are assessed in Canvas; no additional notebook response is required.

## 5. Quantitative univariate EDA: hair toxicology quantity

Univariate EDA examines one variable at a time. Some documented variables support quantitative summaries of center, spread, missingness, and distribution shape. A numeric-looking value or pandas dtype is not enough: documentation must establish that arithmetic differences and quantities are meaningful.


### Documentation and demo

We will inspect `su_y_hairtox__rslt__mj__thccooh_qnt` from `su_y_hairtox`. DEAP documents this variable as:

- `type_var`: summary score;
- `type_data`: double;
- `type_level`: ratio; and
- unit: picograms per ten milligrams (`pg/10mg`).

Its scientific role is a quantitative mass-spectrometry result for THCCOOH. The completed demo selects the Series, reports missingness, runs `describe()`, and displays a histogram. Then you will complete the median, IQR, boxplot, and upper-one-percent review flag in the following code cell.


In [ ]:
# Demo: run this completed cell.
tox_qnt_variable = "su_y_hairtox__rslt__mj__thccooh_qnt"
tox_cnf_variable = "su_y_hairtox__rslt__mj__thccooh_cnf"
tox_qnt_series = su_y_hairtox[tox_qnt_variable]

tox_qnt_summary = tox_qnt_series.describe()
tox_qnt_pct_missing = tox_qnt_series.isna().mean() * 100

print("describe() output:")
print(tox_qnt_summary)
print()
print("Percent missing:", round(tox_qnt_pct_missing, 1))

plt.figure(figsize=(6, 4))
plt.hist(tox_qnt_series.dropna(), bins=30)
plt.xlabel("THCCOOH quantity (pg/10mg)")
plt.ylabel("Number of records")
plt.title("Distribution of hair toxicology quantity")
plt.show()

In [ ]:
# Your Turn: complete the summaries and review flag below.

# TODO: Calculate the median.
tox_qnt_median = ______

# TODO: Calculate Q1 and Q3.
tox_qnt_q1 = ______
tox_qnt_q3 = ______

# TODO: Calculate the IQR.
tox_qnt_iqr = ______

# TODO: Calculate the upper 1 percent review cutoff.
tox_qnt_upper_1pct = ______

print("Median:", tox_qnt_median)
print("Q1:", tox_qnt_q1)
print("Q3:", tox_qnt_q3)
print("IQR:", tox_qnt_iqr)
print("Upper 1% review cutoff:", tox_qnt_upper_1pct)

plt.figure(figsize=(6, 2.5))
# TODO: Create a horizontal boxplot using the nonmissing quantity values.
plt.boxplot(______, vert=False)
plt.xlabel("THCCOOH quantity (pg/10mg)")
plt.title("Boxplot of hair toxicology quantity")
plt.show()

# TODO: Complete the cutoff and sort-variable blanks.
# Keep participant ID, session ID, confirmation, and quantity visible.
tox_qnt_flagged = su_y_hairtox.loc[
    su_y_hairtox[tox_qnt_variable] >= ______,
    [
        "participant_id",
        "session_id",
        tox_cnf_variable,
        tox_qnt_variable,
    ],
].sort_values(______, ascending=False)

display(tox_qnt_flagged.head(10))

### What to notice

DEAP documentation—not the pandas dtype alone—supports treating the quantity as a ratio-level quantitative variable.

`describe()` and the histogram orient us to the observed distribution and missingness. The median and IQR are often useful when the distribution is skewed.

A histogram shows distribution shape. A boxplot shows the median, spread, and potential extreme values.

The upper 1 percent is a **RESEARCHER DECISION** used as a review flag, not a cleaning rule. A high laboratory quantity may be uncommon and still be valid.

Section 6 uses documentation, confirmation status, and external context to ask whether flagged values are merely unusual or potentially suspicious.

### Prepare for the Canvas reflection: quantitative univariate EDA

Use your completed summaries, histogram, boxplot, and flagged-record table to prepare for these questions:

1. What are the median, IQR, percent missing, and upper-one-percent review cutoff for the toxicology quantity variable?
2. What information does the histogram show that the boxplot does not emphasize, and what information does the boxplot make easier to see?
3. Why does DEAP's ratio-level classification support quantitative summaries, while a numeric pandas dtype by itself would not be sufficient?
4. An analyst says, “Every value above the upper-one-percent cutoff is an error and should be deleted.” What is wrong with that conclusion?
5. What does the upper-one-percent rule establish, and what additional evidence would be needed before making a cleaning decision?

These questions are assessed in Canvas; no additional notebook response is required.

## 6. Introductory outlier identification

In the previous section, we described the hair toxicology quantity distribution and used an upper-one-percent rule to flag unusually high values. A value may be unusual because it is high or low relative to the rest of the dataset. That does not automatically mean it is an error.

Hair toxicology quantity requires careful interpretation because the number comes from a laboratory process, not from a direct count of use events.

The main distinction is:

> **A value can be unusual without being suspicious.**

To decide whether a value is suspicious, we need more than a plot. We need the data distribution, documentation, confirmation status, and sometimes external scientific context. We also need to distinguish documented facts from researcher-defined review choices.

### Hair toxicology quantity: unusual versus suspicious values

Now apply outlier review to `su_y_hairtox__rslt__mj__thccooh_qnt`.

A high toxicology quantity value should be interpreted using both the data distribution and the measurement context. The value is a laboratory quantity, not a direct count of cannabis-use days. Hair toxicology can provide biological evidence of exposure across a broad detection window, but it does not identify exact timing, dose, intent, impairment, or frequency of use.

In this lab, we label the evidence supporting each step:

| Review question | Evidence label | Evidence used | What it helps us decide |
|---|---|---|---|
| Is the value unusual in this dataset? | **RESEARCHER DECISION** | Histogram, boxplot, quantiles, and the chosen upper 1% review cutoff | Whether the value stands apart from other observed values under the stated rule |
| Is the value coherent with the measure? | **DEAP/DOC** | Definition, units, confirmation status, special codes, collection, and testing procedures | Whether the value fits how the measure was supposed to be produced |
| Is the value plausible in broader context? | **LITERATURE** | Studies using comparable analytes, units, specimens, and methods | Whether the value seems biologically or technically plausible beyond this dataset |

In Lab 2, we focus mostly on the first two questions. We identify values that are high relative to this dataset, then check whether they occur with a positive confirmation result.

We do **not** remove, cap, or recode values in Lab 2. The upper 1 percent is a researcher-defined review threshold, not an ABCD cleaning rule. The goal is to flag values for review and explain what additional information would be needed before making a cleaning decision.

### Continue the review

Use the outputs you created in Section 5. Examine the flagged quantity records alongside `su_y_hairtox__rslt__mj__thccooh_cnf`, then use the documentation and literature below to decide whether the values are merely unusual or require further investigation. No additional code is required in this section.

### External benchmark: using LITERATURE as context

The flagged values are high relative to this synthetic dataset. To decide whether they are suspicious rather than merely unusual, we can compare them with external evidence from a published hair toxicology study.

Lo Faro et al. (2022) measured cannabinoids and metabolites in hair samples from four adults treated with medical cannabis. For the analyte closest to our lab variable, THC-COOH / THCCOOH, detected values ranged from **0.025 to 0.84 ng/mg**. Because **1 ng = 1,000 pg**, this equals **25 to 840 pg/mg**. The synthetic lab quantity is documented in `pg/10mg`, so the unit-matched range is **250 to 8,400 pg/10mg**.

This benchmark is useful but imperfect. It uses hair samples and a closely related analyte, but it comes from adults treated with medical cannabis, not adolescents in a developmental research study.

For this lab, label this benchmark as **LITERATURE** and use it as context, not as a cleaning rule. If a value is high relative to this dataset but has a positive confirmation result and is not far outside published hair-toxicology values, it may be unusual but not automatically suspicious. A value would become more suspicious if it had unclear units, conflicted with confirmation status, appeared in an impossible range, or was far outside comparable published ranges.

**Source:** Lo Faro, A. F., Venanzi, B., Pilli, G., Ripani, U., Basile, G., Pichini, S., & Busardò, F. P. (2022). Ultra-high-performance liquid chromatography-tandem mass spectrometry assay for quantifying THC, CBD and their metabolites in hair: Application to patients treated with medical cannabis. *Journal of Pharmaceutical and Biomedical Analysis, 217*, 114841. https://doi.org/10.1016/j.jpba.2022.114841

### Prepare for the Canvas reflection: unusual versus suspicious values

Use the hair toxicology quantity example to prepare for these questions:

1. What is the difference between a value being **unusual** and a value being **suspicious**?
2. What can the histogram, boxplot, and upper-one-percent cutoff establish about the flagged values?
3. Why do confirmation status, units, assay documentation, and special-value documentation matter when deciding whether a value is suspicious?
4. How should the Lo Faro et al. benchmark affect your interpretation? Consider whether the analyte, units, specimen, population, and laboratory method are sufficiently comparable and whether an apparent discrepancy should prompt a unit or scaling check.
5. Which claims come from **DEAP/DOC**, **LITERATURE**, and the **RESEARCHER DECISION** to use an upper-one-percent review threshold?
6. Based on the available evidence, should the flagged records be automatically deleted, retained without question, or flagged for further review? Explain.

These questions are assessed in Canvas; no additional notebook response is required.

## 7. Choosing a visualization for two variables

Bivariate EDA asks how two variables look together. Documentation should establish each variable's role and analytic treatment before the variables are compared. Here we examine one categorical × quantitative combination within a single table; joins and merges remain part of Lab 3.


### Demo

Both `su_y_hairtox__rslt__mj__thccooh_cnf` and `su_y_hairtox__rslt__mj__thccooh_qnt` are stored in `su_y_hairtox`. Documentation supports treating the confirmation result as categorical and the documented laboratory quantity as quantitative. Group summaries and side-by-side boxplots let us compare the quantity distribution across confirmation categories without merging tables.


In [ ]:
hairtox_cnf_var = "su_y_hairtox__rslt__mj__thccooh_cnf"
hairtox_qnt_var = "su_y_hairtox__rslt__mj__thccooh_qnt"

hairtox_compare = su_y_hairtox[[hairtox_cnf_var, hairtox_qnt_var]]

hairtox_group_summary = hairtox_compare.groupby(hairtox_cnf_var, dropna=False)[hairtox_qnt_var].describe()
display(hairtox_group_summary)

hairtox_plot_groups = []
hairtox_plot_labels = []
for group_value in hairtox_compare[hairtox_cnf_var].fillna("Missing").unique():
    group_series = hairtox_compare.loc[
        hairtox_compare[hairtox_cnf_var].fillna("Missing") == group_value,
        hairtox_qnt_var,
    ].dropna()
    if len(group_series) > 0:
        hairtox_plot_groups.append(group_series)
        hairtox_plot_labels.append(str(group_value))

plt.figure(figsize=(7, 4))
plt.boxplot(hairtox_plot_groups, tick_labels=hairtox_plot_labels)
plt.title("Toxicology quantity by confirmation result")
plt.xlabel("Confirmation result")
plt.ylabel("THCCOOH quantity")
plt.show()


### What to notice

This comparison is exploratory. It helps us see how a numeric laboratory value is distributed within observed confirmation-result groups.

It does not turn toxicology into ground truth, and it does not tell us dose, exact timing, intent, impairment, or exact frequency of use by itself.

Documentation is still needed to understand response categories, units, laboratory thresholds, and measurement limits. Group differences are descriptive patterns, not proof of causation or measurement accuracy.

| Variable combination | Appropriate initial display | Course location |
|---|---|---|
| Categorical × quantitative | Group summaries and side-by-side boxplots | Lab 2 |
| Categorical × categorical | Contingency table and grouped or stacked bars | Lab 3 |
| Quantitative × quantitative | Scatterplot | Labs 4 & 5 |

Confirmation is more directly aligned than quantity with the binary concordance question in Lab 3. That does not make confirmation universally more reliable, and neither measure is ground truth. Scatterplots are deferred until a later lab can establish two documented quantitative variables, a valid join, a row unit, and an analytic denominator.


### Prepare for the Canvas reflection: choosing a visualization

The Canvas reflection will use short scenarios to assess variable and visualization selection. Prepare for these five consolidated questions:

1. For one documented ratio-level toxicology quantity variable, when would you choose a histogram and when would you choose a boxplot?
2. Which summaries and visualization are appropriate for confirmation status × toxicology quantity, and why?
3. Which display is appropriate for self-report Yes/No × hair Positive/Negative, and what four combinations should remain visible?
4. A scatterplot generally fits two quantitative variables. Why would a use-days × toxicology-quantity scatterplot still require cautious interpretation and fail to establish reporting accuracy?
5. Why is confirmation better aligned than quantity with a binary concordance question, and why does that not make confirmation ground truth?

A technically appropriate plot is not automatically scientifically informative. Documentation is still needed to establish variable meaning, response categories, units, reference periods, selection processes, missingness, row unit, and the limits of interpretation.

These questions are assessed in Canvas; no additional notebook response is required.

### Bridge to Lab 3

Lab 1 introduced ABCD variable-name structure and separated pandas representation from scientific meaning. Exercise 2 introduced documented metadata, raw versus derived variables, score provenance, and bounded interpretation. In Lab 2, we used those distinctions to justify categorical and quantitative univariate EDA and one categorical × quantitative comparison without joining tables.

In Lab 3, we will reconstruct and validate the MAPI score before analyzing it, decide how unavailable values should be represented, join participant-session tables, define a valid comparison denominator, and introduce categorical × categorical bivariate analysis using self-report and hair confirmation. A later lab will introduce scatterplots for two quantitative variables after the row unit, join, and denominator have been justified.


## 8. Finish the lab and complete the Canvas reflection

Complete the required DEAP record and code cells, then run the notebook from top to bottom and check that the requested outputs are visible. Use the displayed outputs to complete the Canvas reflection quiz.

Sections 3–7 preview the reflection questions assessed in Canvas. No additional written reflection responses are required in the notebook, and the notebook itself is not collected.

Complete the corresponding Canvas quiz, which assesses official ABCD documentation lookup, response options and missingness, measurement levels, interpretation of special codes, measure limitations, unusual-versus-suspicious reasoning, and selection of appropriate univariate and bivariate visualizations.